# Conditional operations: 
1. Fast reset
2. T-gate teleportation
3. Repetition code 

Three feed-forward circuits written with the conditional API (`gates.Switch`,
`gates.IfElse`) of `CQT-qibo`'s `eo-switch-native-execution` branch.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "CQT-qibo" / "src"))  # checked-out CQT-qibo

import numpy as np
import qibo
from qibo import Circuit, gates

qibo.set_backend("numpy")

[Qibo 0.3.6|INFO|2026-10-21 18:27:32]: Using numpy backend on /CPU:0


## Fast reset

A single-round active reset: measure a qubit mid-circuit and apply `X` only if
the outcome is 1, leaving it in $|0\rangle$ ideally. Preparing $|+\rangle$
first exercises both feedback outcomes; the second measurement checks the
reset. Actual feedback speed and reset fidelity must be measured on hardware.

In [5]:
c = Circuit(1)
c.add(gates.H(0))  # prepare |+> so both feedback outcomes occur

before_reset = c.add(gates.M(0, collapse=True, register_name="before_reset"))

c.add(gates.IfElse(
    before_reset.is_one(), 
    then_do=gates.X(0)))
c.add(gates.M(0, register_name="after_reset"))

c.draw()
result = c(nshots=1000)
print("Before reset:", before_reset.frequencies())
print("After reset:", result.frequencies())  # ideally all 0


0: ─H─M─if[─X─][─]─M─
Before reset: Counter({'0': 505, '1': 495})
After reset: Counter({'0': 1000})


## T-gate teleportation

Qubit 0 holds the input $|+\rangle$ and qubit 1 the magic state $T|+\rangle$. After the
CNOT, measuring qubit 1 leaves $T|+\rangle$ on qubit 0 for outcome 0 and $T^\dagger|+\rangle$
for outcome 1, which the conditional $S$ corrects ($S\,T^\dagger = T$). Measuring
$T|+\rangle$ in the X basis gives $P(1) = \sin^2(\pi/8) \approx 0.146$.

In [3]:
c = Circuit(2)

c.add(gates.H(0))

c.add(gates.H(1))
c.add(gates.T(1))
c.add(gates.CNOT(0, 1))
t = c.add(gates.M(1, collapse=True))
c.add(gates.IfElse(t.is_one(), then_do=gates.S(0)))
c.add(gates.H(0))
c.add(gates.M(0))

c.draw()
nshots = 10000
p1 = c(nshots=nshots).frequencies()["1"] / nshots
print(f"P(1) = {p1:.3f}, expected sin^2(pi/8) = {np.sin(np.pi / 8) ** 2:.3f}")

0: ─H───o───if[─S─][─]─H─M─
1: ─H─T─X─M────────────────


P(1) = 0.142, expected sin^2(pi/8) = 0.146


## Repetition code

Data qubits 0, 2, 4 and ancillas 1, 3. Each ancilla measures the $ZZ$ parity of its two
neighbours, and a `Switch` on the syndrome flips the data qubit it points to. Bit 0 of
the syndrome (ancilla 1) is the most significant.

In [2]:
data = [0, 2, 4]
anc = [1, 3]
c = Circuit(5)

# Add error manually
c.add(gates.X(data[1]))

c.add(gates.H(anc[0]))
c.add(gates.H(anc[1]))
c.add(gates.CZ(anc[0], data[0]))
c.add(gates.CZ(anc[1], data[1]))
c.add(gates.CZ(anc[0], data[1]))
c.add(gates.CZ(anc[1], data[2]))
c.add(gates.H(anc[0]))
c.add(gates.H(anc[1]))

syndrome = c.add(gates.M(*anc, collapse=True))

c.add(
    gates.Switch(syndrome)
    .case("10", gates.X(data[0]))
    .case("11", gates.X(data[1]))
    .case("01", gates.X(data[2]))
    .default([])  # "00": no error
)

c.add(gates.M(*data))

c.draw()
print(c(nshots=100).frequencies())  # all 000

0: ───Z───────────switch[─X─][───][───][─]─M─
1: ─H─o───o───H─M─|─────────|────|────|──|─|─
2: ─X───Z─Z─────|─switch[───][─X─][───][─]─M─
3: ─H───o───o─H─M─|─────────|────|────|──|─|─
4: ─────────Z─────switch[───][───][─X─][─]─M─


Counter({'000': 100})
